In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pyspark.sql import functions as F

# Check dataset
print("Total records:", f"{cic_df.count():,}")
print("Total columns:", len(cic_df.columns))

# Check required columns
assert "Label" in cic_df.columns, "Label column missing"
assert "_source_file" in cic_df.columns, "Source file column missing"

print("Visualization environment ready!")

In [0]:
label_counts = (
    cic_df
    .groupBy("Label")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

label_counts["Label"] = label_counts["Label"].fillna("MISSING")

fig, ax = plt.subplots(figsize=(13, 8))

plot_data = label_counts.sort_values("count")

bars = ax.barh(
    plot_data["Label"],
    plot_data["count"]
)

ax.set_xscale("log")
ax.set_title("CIC-IDS2017: Traffic Class Distribution",
             fontsize=16, pad=15)
ax.set_xlabel("Number of Network Flows (Log Scale)")
ax.set_ylabel("Traffic Class")
ax.grid(axis="x", alpha=0.3)

for bar, value in zip(bars, plot_data["count"]):
    ax.text(
        value * 1.05,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [0]:
binary_counts = (
    cic_df
    .withColumn(
        "binary_label",
        F.when(
            F.upper(F.trim(F.col("Label"))) == "BENIGN",
            "BENIGN"
        ).otherwise("ATTACK")
    )
    .groupBy("binary_label")
    .count()
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 7))

ax.pie(
    binary_counts["count"],
    labels=binary_counts["binary_label"],
    autopct="%1.2f%%",
    startangle=90,
    wedgeprops={"width": 0.42, "edgecolor": "white"},
    textprops={"fontsize": 12}
)

ax.set_title(
    "CIC-IDS2017: Benign vs Malicious Traffic",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import functions as F
from functools import reduce

BASE_PATH = "/Volumes/workspace/default/cic_ids2017/raw"

files = sorted([
    f.path
    for f in dbutils.fs.ls(BASE_PATH)
    if f.name.lower().endswith(".csv")
])

dataframes = []

for path in files:

    filename = path.split("/")[-1]

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(path)
    )

    # Normalize column names
    cleaned_columns = [
        c.strip().replace("\ufeff", "")
        for c in df.columns
    ]

    df = df.toDF(*cleaned_columns)

    # Preserve source filename
    df = df.withColumn("_source_file", F.lit(filename))

    # Normalize attack labels
    if "Label" in df.columns:
        df = df.withColumn(
            "Label",
            F.trim(F.col("Label"))
        )

    dataframes.append(df)

# Combine all files, preserving differences in columns
cic_df = reduce(
    lambda x, y: x.unionByName(
        y, allowMissingColumns=True
    ),
    dataframes
)

cic_df.createOrReplaceTempView("cic_ids2017")

print("Dataset successfully loaded!")
print("Columns:", len(cic_df.columns))
print("Rows:", f"{cic_df.count():,}")

In [0]:
label_distribution = (
    cic_df
    .groupBy("Label")
    .count()
    .orderBy(F.desc("count"))
)

display(label_distribution)

In [0]:
from pyspark.sql import functions as F

# Multiclass distribution with percentages
total = cic_df.count()

class_distribution = (
    cic_df
    .groupBy("Label")
    .agg(F.count("*").alias("count"))
    .withColumn(
        "percentage",
        F.round(F.col("count") * 100 / total, 4)
    )
    .orderBy(F.desc("count"))
)

display(class_distribution)

In [0]:
binary_df = cic_df.withColumn(
    "binary_label",
    F.when(
        F.upper(F.trim(F.col("Label"))) == "BENIGN",
        "BENIGN"
    ).otherwise("ATTACK")
)

display(
    binary_df.groupBy("binary_label")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
source_counts = (
    cic_df
    .groupBy("_source_file")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

source_counts["file"] = source_counts["_source_file"].str.replace(
    ".pcap_ISCX.csv", "", regex=False
)

fig, ax = plt.subplots(figsize=(13, 7))

plot_data = source_counts.sort_values("count")

bars = ax.barh(
    plot_data["file"],
    plot_data["count"]
)

ax.set_title("Network Traffic Records by Dataset File",
             fontsize=15, pad=15)
ax.set_xlabel("Number of Network Flows")
ax.set_ylabel("Capture Period")
ax.grid(axis="x", alpha=0.3)

for bar, value in zip(bars, plot_data["count"]):
    ax.text(
        value * 1.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [0]:
# Get actual attack counts from CIC-IDS2017
attack_counts = (
    cic_df
    .filter(
        F.col("Label").isNotNull() &
        (F.upper(F.trim(F.col("Label"))) != "BENIGN")
    )
    .groupBy("Label")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .toPandas()
)

# Calculate percentages
attack_percentages = attack_counts.copy()

total_attacks = attack_percentages["count"].sum()

attack_percentages["percentage"] = (
    attack_percentages["count"] / total_attacks * 100
)

attack_percentages = attack_percentages.sort_values(
    "percentage",
    ascending=True
)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    attack_percentages["Label"],
    attack_percentages["percentage"]
)

ax.set_title(
    "Contribution of Each Attack Type to Malicious Traffic",
    fontsize=15,
    pad=15
)

ax.set_xlabel("Percentage of All Malicious Flows (%)")
ax.set_ylabel("Attack Type")
ax.grid(axis="x", alpha=0.3)

for bar, value in zip(
    bars, attack_percentages["percentage"]
):
    ax.text(
        value + 0.1,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

print("Number of attack categories:", len(attack_counts))
print("Total malicious flows:", f"{total_attacks:,}")